In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.utils import to_categorical
import pickle
import numpy as np # linear algebra



In [ ]:
# Charger le fichier CSV
df = pd.read_csv('emotion_dataset_modifie.csv')

# Prétraitement des données
texts = df['Text'].values
labels = df['Emotion'].values

# Encodage des étiquettes
label_encoder = LabelEncoder()
labels_encoded = label_encoder.fit_transform(labels)
labels_categorical = to_categorical(labels_encoded)  # Pour la classification multi-classes

In [3]:
# Division des données
X_train, X_test, y_train, y_test = train_test_split(X, labels_categorical, test_size=0.2, random_state=42)

# Tokenize the text data
tokenizer = Tokenizer(num_words=50000)
tokenizer.fit_on_texts(X_train)
tokenizer.fit_on_texts(X_test)
X_train_sequences = tokenizer.texts_to_sequences(X_train)
X_test_sequences = tokenizer.texts_to_sequences(X_test)


NameError: name 'X' is not defined

In [ ]:
# Max Len in X_train_sequences
maxlen = max(len(tokens) for tokens in X_train_sequences)
print("Maximum sequence length (maxlen):", maxlen)



In [ ]:
# Perform padding on X_train and X_test sequences
X_train_padded = pad_sequences(X_train_sequences, maxlen=maxlen, padding='post',)
X_test_padded = pad_sequences(X_test_sequences, maxlen=maxlen, padding='post')

# Print the padded sequences for X_train and X_test
print("X_train_padded:")
print(X_train_padded)
print("\nX_test_padded:")
print(X_test_padded)



In [ ]:
# Embedding Input Size / Vocabulary Size 
input_Size = np.max(X_train_padded) + 1
input_Size



In [ ]:
# Define the model
model = Sequential()

# Add embedding layer
model.add(Embedding(input_dim=input_Size, output_dim=50, input_length=maxlen))

# Dropout
model.add(Dropout(0.5))

# Add Bidirectional LSTM layer
model.add(Bidirectional(GRU(120, return_sequences=True)))
model.add(Bidirectional(GRU(64, return_sequences=True)))

#Batch Normalization
model.add(BatchNormalization())

# Add Bidirectional GRU layer
model.add(Bidirectional(GRU(64)))

# Add output layer
model.add(Dense(6, activation='softmax'))

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Print model summary
model.summary()

# Entraînement du modèle
history = model.fit(X_train, y_train, epochs=5, batch_size=32, validation_split=0.1)

# Évaluation du modèle
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Model Accuracy: {accuracy}")

# Sauvegarder le modèle et les outils de prétraitement
with open('text_emotion_rnn.pkl', 'wb') as file:
    pickle.dump({
        'model': model,
        'tokenizer': tokenizer,
        'label_encoder': label_encoder
    }, file)

print("Modèle et outils de prétraitement sauvegardés dans 'text_emotion_rnn.pkl'")

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense, Dropout
from tensorflow.keras.utils import to_categorical
import pickle

# Charger le fichier CSV
df = pd.read_csv('emotion_dataset_modifie.csv')

# Prétraitement des données
texts = df['Text'].values
labels = df['Emotion'].values

# Encodage des étiquettes
label_encoder = LabelEncoder()
labels_encoded = label_encoder.fit_transform(labels)
labels_categorical = to_categorical(labels_encoded)  # Pour la classification multi-classes

# Tokenisation du texte
tokenizer = Tokenizer(num_words=20000)  # Augmentez le vocabulaire
tokenizer.fit_on_texts(texts)
X = tokenizer.texts_to_sequences(texts)
X = pad_sequences(X, maxlen=150)  # Ajustez maxlen selon vos besoins

# Division des données
X_train, X_test, y_train, y_test = train_test_split(X, labels_categorical, test_size=0.2, random_state=42)





In [9]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns
# Word Cloud

# from textacy import preprocessing
from nltk.stem.snowball import SnowballStemmer
from keras.preprocessing import sequence
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
import re
# Define the model
model = Sequential()

# Add embedding layer
model.add(Embedding(input_dim=50000, output_dim=50, input_length=150))

# Dropout
model.add(Dropout(0.5))

# Add Bidirectional LSTM layer
model.add(Bidirectional(GRU(120, return_sequences=True)))
model.add(Bidirectional(GRU(64, return_sequences=True)))

#Batch Normalization
model.add(BatchNormalization())

# Add Bidirectional GRU layer
model.add(Bidirectional(GRU(64)))

# Add output layer
model.add(Dense(7, activation='softmax'))

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Print model summary
model.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_2 (Embedding)     (None, 150, 50)           2500000   
                                                                 
 dropout_2 (Dropout)         (None, 150, 50)           0         
                                                                 
 bidirectional_3 (Bidirectio  (None, 150, 240)         123840    
 nal)                                                            
                                                                 
 bidirectional_4 (Bidirectio  (None, 150, 128)         117504    
 nal)                                                            
                                                                 
 batch_normalization_1 (Batc  (None, 150, 128)         512       
 hNormalization)                                                 
                                                      

In [11]:
# Entraînement du modèle
history = model.fit(X_train, y_train, epochs=5, batch_size=1500, validation_data=(X_test, y_test))

# Évaluation du modèle
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Model Accuracy: {accuracy}")

# Sauvegarder le modèle et les outils de prétraitement
with open('text_emotion_rnn_optimized.pkl', 'wb') as file:
    pickle.dump({
        'model': model,
        'tokenizer': tokenizer,
        'label_encoder': label_encoder
    }, file)

print("Modèle et outils de prétraitement sauvegardés dans 'text_emotion_rnn_optimized.pkl'")

Epoch 1/5


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, GRU, Dense, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import pickle

# Charger le fichier CSV
df = pd.read_csv('text_emotion_dataset.csv')

# Prétraitement des données
texts = df['Text'].values
labels = df['Emotion'].values

# Encodage des étiquettes
label_encoder = LabelEncoder()
labels_encoded = label_encoder.fit_transform(labels)
labels_categorical = to_categorical(labels_encoded)

# Tokenisation du texte
tokenizer = Tokenizer(num_words=10000)  # Ajustez num_words selon vos besoins
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
X = pad_sequences(sequences, padding='post')  # Padding des séquences

# Division des données
X_train, X_test, y_train, y_test = train_test_split(X, labels_categorical, test_size=0.2, random_state=42)

# Création du modèle
model = Sequential()
model.add(Embedding(input_dim=10000, output_dim=128, input_length=X.shape[1]))

# Dropout
model.add(Dropout(0.4))  # Réduit le taux de Dropout

# Ajouter une couche Bidirectionnelle GRU
model.add(Bidirectional(GRU(120, return_sequences=True)))
model.add(Bidirectional(GRU(64, return_sequences=True)))

# Normalisation par lots
model.add(BatchNormalization())

# Ajouter une autre couche Bidirectionnelle GRU
model.add(Bidirectional(GRU(64)))

# Ajouter la couche de sortie
model.add(Dense(7, activation='softmax'))  # 6 classes pour les émotions

# Compilation du modèle avec un taux d'apprentissage spécifique

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Callbacks pour le contrôle d'arrêt et l'enregistrement du meilleur modèle
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
model_checkpoint = ModelCheckpoint('best_model.h5', save_best_only=True, monitor='val_loss')

# Entraînement du modèle
model.fit(X_train, y_train, epochs=20, batch_size=32, validation_data=(X_test, y_test),
          callbacks=[early_stopping, model_checkpoint])  # Augmentez le nombre d'époques

# Évaluation du modèle
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Model Accuracy: {accuracy}")

# Sauvegarder le modèle et les outils de prétraitement
model.save('text_emotionMELD_BiGRU_final.h5')
with open('text_emotionMELD_preprocessing.pkl', 'wb') as file:
    pickle.dump({
        'tokenizer': tokenizer,
        'label_encoder': label_encoder
    }, file)

print("Modèle et outils de prétraitement sauvegardés.")

Epoch 1/20
867/867 [==============================] - 811s 923ms/step - loss: 1.2593 - accuracy: 0.5441 - val_loss: 1.0497 - val_accuracy: 0.6364
Epoch 2/20
867/867 [==============================] - 1021s 1s/step - loss: 0.9108 - accuracy: 0.6833 - val_loss: 1.0159 - val_accuracy: 0.6457
Epoch 3/20
867/867 [==============================] - 974s 1s/step - loss: 0.7508 - accuracy: 0.7398 - val_loss: 1.0607 - val_accuracy: 0.6444
Epoch 4/20
867/867 [==============================] - 1246s 1s/step - loss: 0.6404 - accuracy: 0.7826 - val_loss: 1.1227 - val_accuracy: 0.6394
Epoch 5/20
217/217 [==============================] - 68s 315ms/step - loss: 1.0159 - accuracy: 0.6457
Model Accuracy: 0.6457431316375732
Modèle et outils de prétraitement sauvegardés.
